# 13 prepare copykat microbatches

**Purpose:** Prepare CopyKAT microbatches and generate R runner scripts.

**Original analysis notebook:** `13_prepare_CopyKAT_microbatch_all_cells_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 31b Prepare CopyKAT micro-batch inputs — all cells

Safer replacement for the slow/stuck 8k batches.

Batch design: **2000 target cells + 1000 reference anchor cells** per CopyKAT run.

Output:
`${GLIOMA_PROJECT_ROOT}/RAW/integrated/CNV_inputs/CopyKAT_HVG8000_r0_6/batchwise_micro_all_cells`


In [ ]:

import os, gzip, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.io import mmwrite

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
OUT_BASE_DIR = PROJECT_DIR / "data/interim/integrated/CNV_inputs/CopyKAT_HVG8000_r0_6/batchwise_micro_all_cells"
OUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD_CANDIDATES = [
    PROJECT_DIR / "data/processed/integrated/GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad",
    PROJECT_DIR / "data/processed/integrated/GBM_core_4datasets_fullgene_marker_based_annotation.h5ad",
    PROJECT_DIR / "data/processed/integrated/GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad",
]
GENE_ORDER_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"

CLUSTER_KEY = "leiden_scvi_main"
REFERENCE_CLUSTERS = {"8", "14"}
TARGET_CHUNK_SIZE = 2000
REF_ANCHOR_SIZE = 1000
RANDOM_SEED = 42
COPYKAT_N_CORES = 1
BATCH_TIMEOUT_HOURS = 8

def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError("None found:\n" + "\n".join(map(str, paths)))

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def gzip_file(src, dst):
    with open(src, "rb") as f_in, gzip.open(dst, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

INPUT_H5AD = first_existing(INPUT_H5AD_CANDIDATES)
print("[READ]", INPUT_H5AD)
adata = sc.read_h5ad(INPUT_H5AD)
print(adata)
print("counts layer:", "counts" in adata.layers)

if CLUSTER_KEY not in adata.obs.columns:
    raise KeyError(f"{CLUSTER_KEY} not found in adata.obs")

counts = adata.layers["counts"] if "counts" in adata.layers else adata.X
counts_source = "adata.layers['counts']" if "counts" in adata.layers else "adata.X"
if sp.issparse(counts):
    counts = counts.tocsr()
print("counts source:", counts_source, counts.shape)


In [ ]:

gene_order = pd.read_csv(GENE_ORDER_TSV, sep="\t")
gene_order.columns = [str(c).strip() for c in gene_order.columns]

gene_col = next((c for c in ["gene", "gene_symbol", "symbol", "Gene"] if c in gene_order.columns), None)
chr_col = next((c for c in ["chr", "chromosome", "cnv_chr"] if c in gene_order.columns), None)
start_col = next((c for c in ["start", "cnv_start"] if c in gene_order.columns), None)
end_col = next((c for c in ["end", "cnv_end"] if c in gene_order.columns), None)

if any(x is None for x in [gene_col, chr_col, start_col, end_col]):
    raise ValueError(f"Cannot identify gene order columns: {list(gene_order.columns)}")

gene_order = gene_order[[gene_col, chr_col, start_col, end_col]].copy()
gene_order.columns = ["gene", "chr", "start", "end"]
gene_order["gene"] = gene_order["gene"].astype(str)
gene_order["chr"] = gene_order["chr"].astype(str).str.replace("^chr", "", regex=True)
gene_order["start"] = pd.to_numeric(gene_order["start"], errors="coerce")
gene_order["end"] = pd.to_numeric(gene_order["end"], errors="coerce")
gene_order = gene_order.dropna(subset=["gene", "chr", "start", "end"]).drop_duplicates("gene")

var_names = pd.Index(adata.var_names.astype(str))
gene_to_idx = pd.Series(np.arange(adata.n_vars), index=var_names)
matched = gene_order[gene_order["gene"].isin(var_names)].copy()
matched["gene_idx"] = matched["gene"].map(gene_to_idx).astype(int)

chr_order = {str(i): i for i in range(1, 23)}
chr_order.update({"X": 23, "Y": 24, "M": 25, "MT": 25, "mitochondria": 25})
matched["chr_order"] = matched["chr"].map(chr_order).fillna(99).astype(int)
matched = matched.sort_values(["chr_order", "start", "end", "gene"], kind="mergesort").reset_index(drop=True)

usable_gene_idx = matched["gene_idx"].to_numpy(dtype=int)
usable_genes = matched["gene"].astype(str).to_numpy()

report = pd.DataFrame([{
    "input_h5ad": str(INPUT_H5AD),
    "gene_order_file": str(GENE_ORDER_TSV),
    "counts_source": counts_source,
    "n_cells": adata.n_obs,
    "n_genes_total": adata.n_vars,
    "n_genes_usable": len(usable_gene_idx),
    "target_chunk_size": TARGET_CHUNK_SIZE,
    "ref_anchor_size": REF_ANCHOR_SIZE,
}])
write_tsv_replace(report, METADATA_DIR / "copykat_microbatch_gene_position_report_HVG8000_r0_6.tsv")
display(report)


In [ ]:

rng = np.random.default_rng(RANDOM_SEED)
n_cells = adata.n_obs
all_indices = np.arange(n_cells)

cluster_values = adata.obs[CLUSTER_KEY].astype(str).values
ref_pool = np.where(pd.Series(cluster_values).isin(REFERENCE_CLUSTERS).values)[0]
if len(ref_pool) == 0:
    raise ValueError(f"No reference cells found in clusters {REFERENCE_CLUSTERS}")

ref_anchor_n = min(REF_ANCHOR_SIZE, len(ref_pool))
ref_anchor_indices = np.sort(rng.choice(ref_pool, size=ref_anchor_n, replace=False))

target_chunks = [all_indices[i:i + TARGET_CHUNK_SIZE] for i in range(0, n_cells, TARGET_CHUNK_SIZE)]

rows = []
for i, target_idx in enumerate(target_chunks, start=1):
    batch_idx = np.unique(np.concatenate([target_idx, ref_anchor_indices]))
    rows.append({
        "batch_id": f"copykat_microbatch_{i:04d}",
        "batch_num": i,
        "target_start_index": int(target_idx[0]),
        "target_end_index_exclusive": int(target_idx[-1] + 1),
        "n_target_cells": int(len(target_idx)),
        "n_reference_anchor_cells_requested": int(ref_anchor_n),
        "n_cells_exported_unique": int(len(batch_idx)),
        "batch_dir": str(OUT_BASE_DIR / f"copykat_microbatch_{i:04d}"),
        "r_script": str(OUT_BASE_DIR / f"copykat_microbatch_{i:04d}" / f"run_copykat_microbatch_{i:04d}.R"),
        "expected_prediction": str(OUT_BASE_DIR / f"copykat_microbatch_{i:04d}" / f"copykat_microbatch_{i:04d}_prediction.tsv"),
    })
batch_plan = pd.DataFrame(rows)
write_tsv_replace(batch_plan, METADATA_DIR / "copykat_microbatch_batch_plan_HVG8000_r0_6.tsv")
display(batch_plan.head())
display(batch_plan.tail())
print("n batches:", len(batch_plan), "reference anchors:", ref_anchor_n)


In [ ]:

R_TEMPLATE = """
options(stringsAsFactors = FALSE)
options(timeout = 1000)

suppressPackageStartupMessages({{
  library(Matrix)
}})

if (!requireNamespace("copykat", quietly = TRUE)) {{
  stop("The R package 'copykat' is not installed.")
}}

batch_id <- "{batch_id}"
base_dir <- "{batch_dir}"

mtx_gz_file <- file.path(base_dir, "copykat_raw_counts_genes_by_cells.mtx.gz")
mtx_file <- file.path(base_dir, "copykat_raw_counts_genes_by_cells.mtx")
genes_file <- file.path(base_dir, "copykat_genes.tsv")
cells_file <- file.path(base_dir, "copykat_barcodes.tsv")
meta_file <- file.path(base_dir, "copykat_cell_metadata.tsv")

out_prefix <- file.path(base_dir, batch_id)
out_tsv <- paste0(out_prefix, "_prediction.tsv")
out_rds <- paste0(out_prefix, "_prediction.rds")
out_obj <- paste0(out_prefix, "_full_object.rds")
out_info <- paste0(out_prefix, "_run_info.tsv")

message("[BATCH] ", batch_id)

if (file.exists(out_tsv)) {{
  message("[SKIP] prediction exists: ", out_tsv)
  quit(save = "no", status = 0)
}}

if (!file.exists(mtx_file)) {{
  if (!file.exists(mtx_gz_file)) stop("Missing MTX input: ", mtx_gz_file)
  message("[DECOMPRESS] ", mtx_gz_file, " -> ", mtx_file)
  cmd <- sprintf("gzip -dc %s > %s", shQuote(mtx_gz_file), shQuote(mtx_file))
  status <- system(cmd)
  if (!identical(status, 0L)) stop("gzip decompression failed: ", cmd)
}}

if (!file.exists(mtx_file) || file.info(mtx_file)$size == 0) {{
  stop("MTX file missing or empty: ", mtx_file)
}}

genes <- read.delim(genes_file, header = FALSE, stringsAsFactors = FALSE)[[1]]
cells <- read.delim(cells_file, header = FALSE, stringsAsFactors = FALSE)[[1]]
meta <- read.delim(meta_file, header = TRUE, sep = "\\t", stringsAsFactors = FALSE, check.names = FALSE)

raw_sparse <- Matrix::readMM(mtx_file)
rownames(raw_sparse) <- make.unique(as.character(genes))
colnames(raw_sparse) <- make.unique(as.character(cells))
message("[SPARSE] genes x cells: ", nrow(raw_sparse), " x ", ncol(raw_sparse))

rawmat <- as.matrix(raw_sparse)
storage.mode(rawmat) <- "numeric"
message("[DENSE GB] ", round(as.numeric(object.size(rawmat)) / 1024^3, 3))

norm_cells <- meta$cell_id[meta$is_reference_anchor == TRUE | meta$is_reference_anchor == "TRUE"]
norm_cells <- intersect(norm_cells, colnames(rawmat))
message("[NORM anchors] ", length(norm_cells))

args <- list(
  rawmat = rawmat,
  id.type = "S",
  ngene.chr = 5,
  win.size = 25,
  KS.cut = 0.1,
  sam.name = batch_id,
  distance = "euclidean",
  n.cores = {n_cores}
)

formal_names <- names(formals(copykat::copykat))
if ("plot.genes" %in% formal_names) args$plot.genes <- "FALSE"
if ("norm.cell.names" %in% formal_names && length(norm_cells) > 0) args$norm.cell.names <- norm_cells

message("[RUN CopyKAT]")
copykat_result <- do.call(copykat::copykat, args)

saveRDS(copykat_result, out_obj)

prediction <- NULL
if (is.list(copykat_result) && "prediction" %in% names(copykat_result)) {{
  prediction <- copykat_result$prediction
}} else if (is.data.frame(copykat_result)) {{
  prediction <- copykat_result
}}

if (is.null(prediction)) {{
  pred_files <- list.files(base_dir, pattern = "prediction.*txt$|prediction.*tsv$|copykat.pred", full.names = TRUE)
  message("[WARNING] prediction object not found. Files: ", paste(pred_files, collapse = ", "))
  info <- data.frame(batch_id = batch_id, status = "no_prediction_object", n_genes = nrow(rawmat), n_cells = ncol(rawmat))
  write.table(info, out_info, sep = "\\t", row.names = FALSE, quote = FALSE)
  quit(save = "no", status = 2)
}}

pred_df <- as.data.frame(prediction, stringsAsFactors = FALSE)
if (!("cell.names" %in% colnames(pred_df))) pred_df <- cbind(cell.names = rownames(pred_df), pred_df)

write.table(pred_df, out_tsv, sep = "\\t", row.names = FALSE, quote = FALSE)
saveRDS(pred_df, out_rds)

info <- data.frame(
  batch_id = batch_id, status = "completed",
  n_genes = nrow(rawmat), n_cells = ncol(rawmat),
  n_norm_cells = length(norm_cells), n_prediction_rows = nrow(pred_df),
  prediction_tsv = out_tsv
)
write.table(info, out_info, sep = "\\t", row.names = FALSE, quote = FALSE)

message("[DONE] ", batch_id)
"""


In [ ]:

def clean_series(vals):
    return pd.Series(vals).astype(str).replace({"nan": "", "None": "", "<NA>": ""}).values

records = []
for _, row in batch_plan.iterrows():
    batch_id = row["batch_id"]
    batch_num = int(row["batch_num"])
    print("[EXPORT]", batch_id)
    batch_dir = Path(row["batch_dir"])
    batch_dir.mkdir(parents=True, exist_ok=True)

    target_idx = np.arange(int(row["target_start_index"]), int(row["target_end_index_exclusive"]))
    batch_idx = np.unique(np.concatenate([target_idx, ref_anchor_indices]))
    is_target = pd.Series(batch_idx).isin(target_idx).values
    is_ref_anchor = pd.Series(batch_idx).isin(ref_anchor_indices).values

    sub = counts[batch_idx, :][:, usable_gene_idx]
    sub_t = sub.T.tocoo() if sp.issparse(sub) else sp.coo_matrix(sub.T)

    mtx = batch_dir / "copykat_raw_counts_genes_by_cells.mtx"
    mtx_gz = batch_dir / "copykat_raw_counts_genes_by_cells.mtx.gz"
    if mtx.exists(): mtx.unlink()
    if mtx_gz.exists(): mtx_gz.unlink()
    mmwrite(str(mtx), sub_t)
    gzip_file(mtx, mtx_gz)

    genes_path = batch_dir / "copykat_genes.tsv"
    cells_path = batch_dir / "copykat_barcodes.tsv"
    meta_path = batch_dir / "copykat_cell_metadata.tsv"

    pd.Series(usable_genes).to_csv(genes_path, sep="\t", index=False, header=False)
    pd.Series(adata.obs_names[batch_idx].astype(str)).to_csv(cells_path, sep="\t", index=False, header=False)

    meta = pd.DataFrame({
        "cell_id": adata.obs_names[batch_idx].astype(str),
        "original_index": batch_idx.astype(int),
        "is_target_cell": is_target.astype(bool),
        "is_reference_anchor": is_ref_anchor.astype(bool),
        "batch_id": batch_id,
    })
    for col in [CLUSTER_KEY, "annotation_marker_based_HVG8000_r0_6", "annotation_marker_based",
                "condition", "core_dataset", "dataset_id", "patient_id", "sample_id", "CNV_role_recommendation"]:
        if col in adata.obs.columns:
            meta[col] = clean_series(adata.obs.iloc[batch_idx][col].values)
    meta.to_csv(meta_path, sep="\t", index=False)

    r_script = batch_dir / f"run_copykat_microbatch_{batch_num:04d}.R"
    r_script.write_text(R_TEMPLATE.format(batch_id=batch_id, batch_dir=str(batch_dir), n_cores=COPYKAT_N_CORES))

    records.append({
        "batch_id": batch_id,
        "batch_num": batch_num,
        "batch_dir": str(batch_dir),
        "n_cells_exported": int(len(batch_idx)),
        "n_target_cells": int(is_target.sum()),
        "n_reference_anchor_cells": int(is_ref_anchor.sum()),
        "n_genes": int(len(usable_genes)),
        "mtx_gz": str(mtx_gz),
        "mtx": str(mtx),
        "genes": str(genes_path),
        "barcodes": str(cells_path),
        "metadata": str(meta_path),
        "r_script": str(r_script),
        "expected_prediction": str(batch_dir / f"{batch_id}_prediction.tsv"),
    })

manifest = pd.DataFrame(records)
write_tsv_replace(manifest, METADATA_DIR / "copykat_microbatch_input_manifest_HVG8000_r0_6.tsv")
display(manifest.head())
display(manifest.tail())


In [ ]:

MASTER_SH = OUT_BASE_DIR / "run_all_copykat_microbatches_HVG8000_r0_6.sh"
manifest_path = METADATA_DIR / "copykat_microbatch_input_manifest_HVG8000_r0_6.tsv"

shell = f'''#!/usr/bin/env bash
set -u
BASE_DIR="{OUT_BASE_DIR}"
MANIFEST="{manifest_path}"
echo "[START] CopyKAT microbatch runner"
cd "${{BASE_DIR}}" || exit 1
while IFS=$'\t' read -r batch_id batch_num batch_dir n_cells_exported n_target_cells n_reference_anchor_cells n_genes mtx_gz mtx genes barcodes metadata r_script expected_prediction; do
  if [[ "${{batch_id}}" == "batch_id" ]]; then continue; fi
  echo "---------------------------------------------"
  echo "[BATCH] ${{batch_id}}"
  if [[ -f "${{expected_prediction}}" ]]; then
    echo "[SKIP] ${{expected_prediction}}"
    continue
  fi
  LOG="${{batch_dir}}/${{batch_id}}_run.log"
  timeout {BATCH_TIMEOUT_HOURS}h Rscript --vanilla "${{r_script}}" > "${{LOG}}" 2>&1
  STATUS=$?
  echo "[STATUS] ${{batch_id}} ${{STATUS}}"
done < "${{MANIFEST}}"
echo "[DONE] CopyKAT microbatch runner"
'''
MASTER_SH.write_text(shell)
os.chmod(MASTER_SH, 0o755)

out = pd.DataFrame([{
    "base_dir": str(OUT_BASE_DIR),
    "master_shell": str(MASTER_SH),
    "manifest": str(manifest_path),
    "n_batches": len(batch_plan),
    "target_chunk_size": TARGET_CHUNK_SIZE,
    "ref_anchor_size": REF_ANCHOR_SIZE,
}])
write_tsv_replace(out, METADATA_DIR / "copykat_microbatch_prepare_output_manifest_HVG8000_r0_6.tsv")
display(out)
print("Run command:")
print(f"cd {OUT_BASE_DIR}")
print(f"nohup bash {MASTER_SH.name} > copykat_microbatch_nohup.log 2>&1 &")
